#### Player Journey Data

**Goal**  
Create realistic mobile game player funnel data for funnel analysis, cohort retention, and behavioral clustering.

**Stages modeled**  
1. Install  
2. Register / Login  
3. Complete Tutorial  
4. Play First Game Session  
5. Make First Purchase (IAP)  
6. Return on Day 1 / Day 7 / Day 30

**Features**  
- Drop-off probabilities per stage (realistic for casual games)  
- Time delays between stages  
- Segments / personas (high spender, casual, churned early, etc.)  
- 50,000 players (good balance: large enough, runs fast)

#### 1. Imports & seed

In [14]:
import pandas as pd
import numpy as np
from datetime import timedelta

np.random.seed(42)  # reproducibility

print("Environment ready")
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)

Environment ready
NumPy: 1.26.3
Pandas: 2.3.3


#### 2. Parameters

In [15]:
N_PLAYERS = 50_000

# Probabilities – realistic for mid-tier casual game (adjust if needed)
P_REGISTER     = 0.92    # 92% of installs register
P_TUTORIAL     = 0.78    # 78% finish tutorial
P_FIRST_GAME   = 0.65    # 65% play at least one game
P_FIRST_PURCHASE = 0.035 # ~3.5% make first IAP (typical F2P)
P_D1_RETURN    = 0.44    # D1 retention baseline
P_D7_RETURN    = 0.22    # D7 retention baseline
P_D30_RETURN   = 0.09    # D30 retention baseline

print(f"Generating data for {N_PLAYERS:,} players")

Generating data for 50,000 players


#### 3. Create base player table

In [16]:
players = pd.DataFrame({
    'player_id': range(1, N_PLAYERS + 1),
    'install_date': pd.date_range(
        start='2023-01-01',  # Non-futuristic start
        end='2025-12-31',    # Ends in 2025
        freq='T'             # Minute frequency for more granular spread
    )[:N_PLAYERS]           # Take first N_PLAYERS
})

# Add random variation to install times (seconds)
players['install_date'] += pd.to_timedelta(np.random.randint(0, 60, N_PLAYERS), unit='s')

print("Base table created")
print("Install date range:", players['install_date'].min().date(), "to", players['install_date'].max().date())
display(players.head(5))

C:\Users\Eldu\AppData\Local\Temp\ipykernel_9172\1592530789.py:3: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  'install_date': pd.date_range(


Base table created
Install date range: 2023-01-01 to 2023-02-04


,player_id,install_date
0,1,2023-01-01 00:00:38
1,2,2023-01-01 00:01:51
2,3,2023-01-01 00:02:28
3,4,2023-01-01 00:03:14
4,5,2023-01-01 00:04:42


#### 4. Simulate funnel progression (binary flags)

In [17]:
# Random decisions for each stage
players['registered']      = np.random.binomial(1, P_REGISTER, N_PLAYERS)
players['tutorial_completed'] = np.where(
    players['registered'] == 1,
    np.random.binomial(1, P_TUTORIAL, N_PLAYERS),
    0
)
players['first_game_played'] = np.where(
    players['tutorial_completed'] == 1,
    np.random.binomial(1, P_FIRST_GAME, N_PLAYERS),
    0
)
players['first_purchase']  = np.where(
    players['first_game_played'] == 1,
    np.random.binomial(1, P_FIRST_PURCHASE, N_PLAYERS),
    0
)

# Retention flags (only for those who reached first game)
players['d1_return'] = np.where(
    players['first_game_played'] == 1,
    np.random.binomial(1, P_D1_RETURN, N_PLAYERS),
    0
)
players['d7_return'] = np.where(
    players['first_game_played'] == 1,
    np.random.binomial(1, P_D7_RETURN, N_PLAYERS),
    0
)
players['d30_return'] = np.where(
    players['first_game_played'] == 1,
    np.random.binomial(1, P_D30_RETURN, N_PLAYERS),
    0
)

print("Funnel flags generated")
display(players[['player_id', 'registered', 'tutorial_completed', 'first_game_played',
                 'first_purchase', 'd1_return', 'd7_return', 'd30_return']].head(8))

Funnel flags generated


,player_id,registered,tutorial_completed,first_game_played,first_purchase,d1_return,d7_return,d30_return
0,1,1,1,1,0,0,0,0
1,2,1,1,1,0,0,0,0
2,3,1,1,0,0,0,0,0
3,4,0,0,0,0,0,0,0
4,5,1,1,1,0,1,0,0
5,6,1,1,1,1,0,0,0
6,7,1,0,0,0,0,0,0
7,8,1,1,0,0,0,0,0


#### 5. Add time deltas (realistic delays between stages)



In [21]:
# Time from install to registration (minutes)
players['time_to_register'] = np.where(
    players['registered'] == 1,
    np.random.exponential(scale=30, size=N_PLAYERS),   # most in first hour
    np.nan
)

# Time from registration to tutorial complete (hours)
players['time_to_tutorial'] = np.where(
    players['tutorial_completed'] == 1,
    np.random.exponential(scale=2, size=N_PLAYERS),     # most within 1–2 days
    np.nan
)

# Time from tutorial to first game (minutes)
players['time_to_first_game'] = np.where(
    players['first_game_played'] == 1,
    np.random.exponential(scale=15, size=N_PLAYERS),
    np.nan
)

# Time from first game to first purchase (days)
players['time_to_first_purchase'] = np.where(
    players['first_purchase'] == 1,
    np.random.exponential(scale=7, size=N_PLAYERS),     # most within first week
    np.nan
)

print("Time deltas added (in minutes/days)")
display(players.filter(like='time_to_').describe().round(1))

Time deltas added (in minutes/days)


,time_to_register,time_to_tutorial,time_to_first_game,time_to_first_purchase
count,45905.0,35902.0,23168.0,811.0
mean,30.0,2.0,14.8,7.0
std,29.9,2.0,15.0,6.9
min,0.0,0.0,0.0,0.0
25%,8.7,0.6,4.3,1.9
50%,20.8,1.4,10.2,5.0
75%,41.8,2.8,20.4,9.7
max,361.5,20.2,167.5,59.0


#### 6. Add revenue & some extra behavioral signals

In [22]:
# Total revenue (skewed – most 0, few high spenders)
players['total_revenue'] = np.where(
    players['first_purchase'] == 1,
    np.random.lognormal(mean=1.5, sigma=1.2, size=N_PLAYERS).round(2),  # $4–$50 typical
    0
)

# Extra signals for clustering later
players['avg_session_length_min'] = np.random.normal(8, 4, N_PLAYERS).clip(1, 30).round(1)
players['total_sessions'] = np.random.poisson(lam=5, size=N_PLAYERS).clip(0)

print("Revenue & behavior signals added")
display(players[['total_revenue', 'avg_session_length_min', 'total_sessions']].describe().round(2))

Revenue & behavior signals added


,total_revenue,avg_session_length_min,total_sessions
count,50000.00,50000.00,50000.00
mean,0.15,8.06,5.01
std,2.25,3.88,2.24
min,0.00,1.00,0.00
25%,0.00,5.30,3.00
50%,0.00,8.00,5.00
75%,0.00,10.70,6.00
max,220.47,26.30,17.00


#### 7. Save the raw dataset



In [23]:
import os

output_path = "../data/raw/player_journey_50k.csv"

# Create directory if not exist
os.makedirs(os.path.dirname(output_path), exist_ok=True)

players.to_csv(output_path, index=False)

print(f"Dataset saved → {output_path}")
print("Shape:", players.shape)
print("\nFirst 5 rows:")
display(players.head(5))

Dataset saved → ../data/raw/player_journey_50k.csv
Shape: (50000, 16)

First 5 rows:


,player_id,install_date,registered,tutorial_completed,first_game_played,first_purchase,d1_return,d7_return,d30_return,total_revenue,avg_session_length_min,total_sessions,time_to_register,time_to_tutorial,time_to_first_game,time_to_first_purchase
0,1,2023-01-01 00:00:38,1,1,1,0,0,0,0,0.0,1.6,5,15.868152,3.015573,20.664393,NaN
1,2,2023-01-01 00:01:51,1,1,1,0,0,0,0,0.0,14.4,5,12.040072,3.454653,18.621251,NaN
2,3,2023-01-01 00:02:28,1,1,0,0,0,0,0,0.0,14.1,3,26.666569,1.643084,NaN,NaN
3,4,2023-01-01 00:03:14,0,0,0,0,0,0,0,0.0,6.6,6,NaN,NaN,NaN,NaN
4,5,2023-01-01 00:04:42,1,1,1,0,1,0,0,0.0,10.8,4,21.676712,1.208063,25.926909,NaN


#### 8. Quick funnel summary (sanity check)

In [24]:
funnel = pd.DataFrame({
    'Stage': [
        'Install',
        'Registered',
        'Tutorial Completed',
        'First Game Played',
        'First Purchase',
        'D1 Return',
        'D7 Return',
        'D30 Return'
    ],
    'Players': [
        N_PLAYERS,
        players['registered'].sum(),
        players['tutorial_completed'].sum(),
        players['first_game_played'].sum(),
        players['first_purchase'].sum(),
        players['d1_return'].sum(),
        players['d7_return'].sum(),
        players['d30_return'].sum()
    ]
})

funnel['Conversion %'] = (funnel['Players'] / N_PLAYERS * 100).round(1)
funnel['Drop-off %'] = 100 - funnel['Conversion %'].shift(-1).fillna(0)

print("Funnel overview:")
display(funnel)

Funnel overview:


,Stage,Players,Conversion %,Drop-off %
0,Install,50000,100.0,8.2
1,Registered,45905,91.8,28.2
2,Tutorial Completed,35902,71.8,53.7
3,First Game Played,23168,46.3,98.4
4,First Purchase,811,1.6,79.5
5,D1 Return,10228,20.5,89.8
6,D7 Return,5077,10.2,95.8
7,D30 Return,2079,4.2,100.0
